In [ ]:
import { RecursiveSet as Set, Tuple, Value } from 'recursive-Set';
type Variable = string;
type Literal  = Variable | Tuple<['¬', Variable]>;
type Clause   = Set<Literal>;

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

The function $\texttt{complement}(l)$ computes the *complement* of a literal $l$.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either 
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [ ]:
function complement(l: Literal): Literal {
    if (typeof l == 'string') { return tpl('¬', l); } 
    return l.get(1);
}

In [ ]:
complement('p');

In [ ]:
complement(tpl('¬', 'p'));

The function $\texttt{extractVariable}(l)$ extracts the propositional variable from the literal $l$.
If $p$ is a propositional variable, we have the following: 

1. $\texttt{extractVariable}(p) = p$,
2. $\texttt{extractVariable}(\neg p) = p$.

In [ ]:
function extractVariable(l: Literal): Variable {
    if (typeof l == 'string') { return l; } 
    return l.get(1);
}

In [ ]:
extractVariable('p');

In [ ]:
extractVariable(tpl('¬', 'p'));

The function $\texttt{collectsVariables}(M)$ takes a set of clauses $M$ as its input and computes the set of all propositional variables occurring in $M$.  The clauses in $M$ are represented as sets of literals.

In [ ]:
function collectVariables(M: Set<Clause>): Set<Variable> {
    let result = set<Variable>();
    return result.flatMap(M, cl => cl.map(extractVariable));
}

In [ ]:
const C1: Clause = set('p', 'q', 'r');
const C2: Clause = set(tpl('¬', 'p'), tpl('¬', 'q'), tpl('¬', 's'));
const M = set(C1, C2);
console.log(collectVariables(M).toString());

The function `prettifyClause(C)` converts the clause `C` into a string. 

In [ ]:
function prettifyClause(clause: Set<Literal>): string {
    const literals = [...clause].map(
        lit => typeof lit == 'string' ? lit : `${lit.get(0)}${lit.get(1)}`
    );
    return `{${literals.join(', ')}}`;
}

The function `prettifyClause(M)` converts the set of clauses `M` into a string.

In [ ]:
function prettify(M: Set<Clause>): string {
    return `{${[...M].map(prettifyClause).join(', ')}}`;
}

Given two clauses $C_1$ and $C_2$ that are represented as sets of literals, the function `cutRule`$(C_1, C_2)$ computes the set of all clauses that can be derived from $C_1$ and $C_2$ using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$

In [ ]:
function cutRule(C1: Clause, C2: Clause): Set<Clause> {
    return C1.filterMap(
        l => C2.has(complement(l)),
        l => {
            const lBar = complement(l);
            // (C1 \ {l}) U (C2 \ {lBar})
            return C1.difference(set(l)).union(C2.difference(set(lBar)));
        }
    );
}

In [ ]:
const C1: Clause = set(tpl('¬', 'q'), tpl('¬', 'r'));
const C2: Clause = set('q', 'r');
prettify(cutRule(C1, C2));

In the expression `saturate(Clauses)` below, `Clauses` is a set of *clauses*, where each clause is a set of *literals*.  The call `saturate(Clauses)` computes the set of all clauses that can be derived from clauses in the set `Clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the set `Clauses` and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D \in \texttt{Clauses}$ or $\{\} \in \texttt{Clauses}$.

In [ ]:
function saturate(Clauses: Set<Clause>): Set<Clause> {
    let currentClauses = Clauses.clone(); 
    while (true) {
        const sizeBefore = currentClauses.size;
        // 1. Generate all possible pairs of clauses (C1, C2)
        const pairs = currentClauses.cartesianProduct(currentClauses);
        // 2. Map each pair through the cutRule, flattening the results into one set
        const derived = set<Clause>().flatMap(pairs, ([C1, C2]) => cutRule(C1, C2));
        if (derived.some(D => D.isEmpty())) {
            return set(set());
        }
        currentClauses = currentClauses.union(derived);
        if (currentClauses.size == sizeBefore) {
            return currentClauses;
        }
    }
}

In [ ]:
const C1: Clause = set('p', 'q');
const C2: Clause = set(tpl('¬', 'p'));
const C3: Clause = set(tpl('¬', 'p'), tpl('¬', 'q'));
const Clauses = set(C1, C2, C3);
prettify(saturate(Clauses));

The function $\texttt{findValuation}(\texttt{Clauses})$ takes a set of clauses as input.  The function tries to compute a variable interpretation that makes all of these clauses `true`.  If this attempt is successful, a set of literals is returned.  This set of literals does not contain  any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If $\texttt{Clauses}$ is unsatisfiable, `false` is returned.

In [ ]:
function findValuation(Clauses: Set<Clause>): false | Set<Literal> {
    const Saturated = saturate(Clauses);
    if (Saturated.some(c => c.isEmpty())) { return false; }        
    const Variables = collectVariables(Clauses);
    const Literals = set<Literal>(); 
    for (const p of Variables) {
        // A clause forces 'p' to be true if it contains 'p', 
        // AND all OTHER literals in the clause evaluate to false.
        // (i.e., their complements are already in our Literals set)
        const forcesP = Saturated.some(
            C => C.has(p) && C.difference(set(p)).every(l => Literals.has(complement(l)))
        );
        Literals.add(forcesP ? p : complement(p));
    }    
    return Literals;
}

In [ ]:
const C01 = set<Literal>('r', 'p', 's');
const C02 = set<Literal>('r', 's');
const C03 = set<Literal>('p', 'q', 's');
const C04 = set<Literal>(tpl('¬', 'p'), tpl('¬', 'q'));
const C05 = set<Literal>(tpl('¬', 'p'), 's', tpl('¬', 'r'));
const C06 = set<Literal>('p', tpl('¬', 'q'), 'r');
const C07 = set<Literal>(tpl('¬', 'r'), tpl('¬', 's'), 'q');
const C08 = set<Literal>(tpl('¬', 'p'), tpl('¬', 's'));
const C09 = set<Literal>('p', tpl('¬', 'r'), tpl('¬', 'q'));
const C10 = set<Literal>(tpl('¬', 'p'), 'r', 'q', tpl('¬', 's'));
const C11 = set<Literal>('p', 'q', 'r', tpl('¬', 's'));

const Clauses1: Set<Clause> = set(
    C01, C02, C03, C04, C05, C06, C07, C08, C09, C10
);

const valuation = findValuation(Clauses1);
if (valuation) {
    console.log(prettifyClause(valuation));
} else {
    console.log("false");
}

In [ ]:
const Clauses2: Set<Clause> = set(
  C01, C02, C03, C04, C05, C06, C07, C08, C09, C10, C11
);
const valuation = findValuation(Clauses2);
if(valuation){
    console.log(prettifyClause(valuation));
} else{
    console.log("false")
}

In [ ]:
saturate(Clauses2);